# Machine Learning on radiomic data — interactive lab

**ICTP College on Medical Physics**

In this lab you build a complete machine-learning pipeline on real clinical
datasets and see how each choice you make changes the result.

You will:

1. pick one of the available **datasets** (each one is a different exercise),
2. decide whether to **rebalance** the training set (SMOTE / ADASYN),
3. assemble a **pipeline**: scaling → feature selection → classifier,
4. let a **grid search** tune the hyperparameters for you,
5. read the **ROC curve, confusion matrix and feature importance**,
6. log your result to the shared class spreadsheet and compare with everyone else.

Nothing is hard-coded: every choice is a menu. Change one thing at a time and
watch what happens — that is the whole point of the exercise.

## Before you start

```text
[ ] You are signed in to Google (the notebook reads and writes a Google Sheet)
[ ] Runtime > Change runtime type > CPU is fine; no GPU needed
[ ] Run the cells in order, top to bottom
[ ] Re-running a cell is always safe
```

If a cell fails, run the cell above it again — usually something further up was
not executed yet.

## Step 1 — Set up and connect to your Google account

This installs the two extra packages Colab does not ship with, imports
everything, and asks permission to read the shared spreadsheet.
A popup will ask you to authorise: accept it.

In [ ]:
#@title ⚙ Step 1 — Install, import, authenticate { display-mode: "form" }

# imbalanced-learn ships with Colab, but we pin the import so the notebook also
# runs on a plain Jupyter install
try:
    import imblearn  # noqa: F401
except ImportError:
    !pip -q install imbalanced-learn
try:
    import xgboost  # noqa: F401
except ImportError:
    !pip -q install xgboost

# --- Google authentication (Colab only) ---------------------------------
IN_COLAB = True
try:
    from google.colab import auth
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    import gspread
    gc = gspread.authorize(creds)
except Exception as e:
    IN_COLAB = False
    gc = None
    print("Not running in Colab (or authentication skipped):", type(e).__name__)

# --- data handling -------------------------------------------------------
import numpy as np
import pandas as pd
import collections, time
from datetime import datetime

# --- plotting ------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# --- machine learning ----------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn import feature_selection
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, recall_score, roc_curve, auc)
import xgboost as xgb

# --- class rebalancing ---------------------------------------------------
from imblearn.over_sampling import SMOTE, ADASYN

# --- widgets -------------------------------------------------------------
import ipywidgets as W
from IPython.display import display, clear_output

SHEET_URL = "https://docs.google.com/spreadsheets/d/1aRtMh_sE22B5-hy8d4VZU52cy2zmSQuLWy1tYGhBPQ8/edit?usp=sharing"

# S holds everything the later steps need, so each cell can be re-run alone
S = {}

print("Setup complete. Colab authentication:", "yes" if IN_COLAB else "no")

## Step 2 — Choose your dataset

The shared spreadsheet holds one worksheet per exercise (the ones whose name
ends in `Data`). The menu below is filled in automatically, so you always see
whatever is actually in the sheet today.

The datasets differ in size, number of features and how balanced the two
classes are — that last point matters a lot in Step 3.

In [ ]:
#@title 📊 Step 2 — Pick a dataset and load it { display-mode: "form" }

sht = gc.open_by_url(SHEET_URL)

# discover the exercises: every worksheet called "<name>Data"
DATASETS = [ws.title[:-4] for ws in sht.worksheets() if ws.title.endswith("Data")]
if not DATASETS:
    raise RuntimeError("No worksheet ending in 'Data' found in the spreadsheet.")

ds_dd = W.Dropdown(options=DATASETS, value=DATASETS[0],
                   description="dataset", style={"description_width": "90px"})
load_btn = W.Button(description="Load dataset", button_style="primary", icon="download")
out2 = W.Output()


def load_dataset(_=None):
    with out2:
        clear_output(wait=True)
        name = ds_dd.value
        ws = sht.worksheet(name + "Data")
        data = pd.DataFrame(ws.get_all_records())
        data = data.ffill()          # forward-fill, the modern spelling of fillna(method='ffill')

        if "outcome" not in data.columns:
            raise KeyError("Worksheet '%sData' has no 'outcome' column." % name)

        Y = data["outcome"]
        X = data.drop("outcome", axis=1)

        S.update(dataset=name, X=X, Y=Y)
        # anything computed downstream is now stale
        for k in ("X_train", "X_test", "Y_train", "Y_test", "fit", "pipe"):
            S.pop(k, None)

        counts = collections.Counter(Y)
        minority = min(counts.values()) / max(1, sum(counts.values()))
        print("Dataset      :", name)
        print("Patients     :", X.shape[0])
        print("Features     :", X.shape[1])
        print("Classes      :", dict(counts),
              " -> minority is %.0f%% of the cohort" % (100 * minority))
        if minority < 0.35:
            print("\nThis dataset is imbalanced. Keep it in mind at Step 3.")
        display(X.head())


load_btn.on_click(load_dataset)
display(W.VBox([W.HBox([ds_dd, load_btn]), out2]))
load_dataset()

## Step 3 — Split, and decide about rebalancing

The test set is held out and **never** rebalanced — it has to keep the class
proportions you would meet in the clinic. Only the *training* set is touched.

| choice | what it does |
|---|---|
| **none** | leave the training set as it is |
| **SMOTE** | invent synthetic minority cases along the line joining close neighbours |
| **ADASYN** | same idea, but concentrates the synthetic cases where the minority class is hardest to classify |

Oversampling usually raises sensitivity and lowers specificity. Try both and
look at what moves.

In [ ]:
#@title ✂ Step 3 — Train/test split and class rebalancing { display-mode: "form" }

test_sl = W.FloatSlider(value=0.30, min=0.15, max=0.40, step=0.05,
                        description="test size", readout_format=".0%",
                        continuous_update=False, style={"description_width": "90px"})
bal_dd = W.Dropdown(options=["none", "SMOTE", "ADASYN"], value="SMOTE",
                    description="rebalance", style={"description_width": "90px"})
seed_txt = W.IntText(value=21, description="seed", style={"description_width": "90px"},
                     layout=W.Layout(width="180px"))
split_btn = W.Button(description="Split & rebalance", button_style="primary", icon="cut")
out3 = W.Output()


def do_split(_=None):
    with out3:
        clear_output(wait=True)
        if "X" not in S:
            print("Load a dataset first (Step 2)."); return

        X_tr, X_te, Y_tr, Y_te = train_test_split(
            S["X"], S["Y"], test_size=test_sl.value,
            random_state=int(seed_txt.value), stratify=S["Y"])

        print("Train set before rebalancing:", dict(collections.Counter(Y_tr)))

        method = bal_dd.value
        if method != "none":
            sampler = (SMOTE(random_state=15) if method == "SMOTE"
                       else ADASYN(random_state=15))
            try:
                X_tr, Y_tr = sampler.fit_resample(X_tr, Y_tr)
                print("Train set after %-7s:" % method, dict(collections.Counter(Y_tr)))
            except ValueError as e:
                # ADASYN refuses to run when the minority class is tiny or already balanced
                print("%s could not be applied (%s)." % (method, e))
                print("Continuing without rebalancing.")
                method = "none"
        else:
            print("Training set left untouched.")

        print("Test set (never rebalanced):", dict(collections.Counter(Y_te)))

        S.update(X_train=X_tr, X_test=X_te, Y_train=Y_tr, Y_test=Y_te,
                 balancing=method, test_size=test_sl.value, seed=int(seed_txt.value))
        S.pop("fit", None)


split_btn.on_click(do_split)
display(W.VBox([W.HBox([test_sl, bal_dd, seed_txt]), split_btn, out3]))
do_split()

## Step 4 — Assemble the pipeline

Three stages, in this order:

1. **scaler** — puts all features on a comparable range. Distance-based models
   (SVM, neural nets) need it; trees do not care.
2. **feature selection** — throws away the features that do not earn their
   place. With a few hundred patients and dozens of features this is what keeps
   you from overfitting.
3. **classifier** — the model itself.

Some combinations are slow: *sequential* feature selection retrains the model
once per candidate feature. Start with `SelectFromModel`.

`Univariate (ANOVA F)` scores each feature on its own, ignoring the model: it is
the fastest option and the only one that treats every classifier equally.

In [ ]:
#@title 🔧 Step 4 — Build the pipeline { display-mode: "form" }

CLASSIFIERS = {
    "Random forest":   lambda: RandomForestClassifier(random_state=0),
    "Decision tree":   lambda: DecisionTreeClassifier(random_state=0),
    "Extra trees":     lambda: ExtraTreesClassifier(random_state=0),
    "XGBoost":         lambda: xgb.sklearn.XGBClassifier(eval_metric="logloss"),
    "SVM":             lambda: SVC(probability=True, random_state=0),
    "Neural network":  lambda: MLPClassifier(hidden_layer_sizes=(15,), max_iter=1000,
                                             random_state=0),
}
SCALERS = {
    "Robust":   RobustScaler,
    "Standard": StandardScaler,
    "none":     None,
}
FSELECT = ["SelectFromModel", "Univariate (ANOVA F)", "RFE",
           "Sequential forward", "Sequential backward"]

clf_dd = W.Dropdown(options=list(CLASSIFIERS), value="Random forest",
                    description="classifier", style={"description_width": "110px"})
sca_dd = W.Dropdown(options=list(SCALERS), value="Robust",
                    description="scaler", style={"description_width": "110px"})
fs_dd = W.Dropdown(options=FSELECT, value="SelectFromModel",
                   description="feature sel.", style={"description_width": "110px"})
build_btn = W.Button(description="Build pipeline", button_style="primary", icon="wrench")
out4 = W.Output()


def build_pipeline(_=None):
    with out4:
        clear_output(wait=True)
        clf = CLASSIFIERS[clf_dd.value]()

        name = fs_dd.value

        # SelectFromModel and RFE rank features using the estimator's own
        # importances. SVM (rbf) and neural networks expose none, so we give the
        # selector its own small forest and keep the chosen model as classifier.
        NEEDS_IMPORTANCE = ("SelectFromModel", "RFE")
        selector_est, note = clf, None
        if name in NEEDS_IMPORTANCE and clf_dd.value in ("SVM", "Neural network"):
            selector_est = RandomForestClassifier(n_estimators=100, random_state=0)
            note = ("%s exposes no feature importances, so the selector ranks features\n"
                    "      with a random forest; %s still does the classifying."
                    % (clf_dd.value, clf_dd.value))

        if name == "SelectFromModel":
            f5 = SelectFromModel(estimator=selector_est)
        elif name == "Univariate (ANOVA F)":
            f5 = feature_selection.SelectKBest(feature_selection.f_classif)
        elif name == "RFE":
            f5 = feature_selection.RFE(estimator=selector_est, step=1)
        else:
            direction = "forward" if "forward" in name else "backward"
            f5 = feature_selection.SequentialFeatureSelector(estimator=clf,
                                                             direction=direction)
        if note:
            print("Note:", note, "\n")

        steps = []
        if SCALERS[sca_dd.value] is not None:
            steps.append(("scaler", SCALERS[sca_dd.value]()))
        steps += [("FS", f5), ("clf", clf)]
        pipe = Pipeline(steps)

        S.update(pipe=pipe, clf=clf, fs=f5,
                 clf_name=clf_dd.value, scaler_name=sca_dd.value, fs_name=name)
        S.pop("fit", None)

        print("Pipeline:")
        for sname, obj in pipe.steps:
            print("   %-8s %s" % (sname, obj.__class__.__name__))
        if name in ("Sequential forward", "Sequential backward"):
            print("\nHeads-up: sequential selection is slow. Be patient at Step 5.")


build_btn.on_click(build_pipeline)
display(W.VBox([W.HBox([sca_dd, fs_dd]), clf_dd, build_btn, out4]))
build_pipeline()

## Step 5 — Tune the hyperparameters

A **grid search** trains the pipeline once for every combination of settings and
keeps the one with the best 5-fold cross-validation score. Cross-validation
happens *inside* the training set, so the test set stays untouched.

The grid adapts to the classifier you picked. More values means a better model
and a longer wait.

In [ ]:
#@title 🔎 Step 5 — Grid search { display-mode: "form" }

nfeat_dd = W.SelectMultiple(options=[3, 5, 10, 15], value=(3, 10),
                            description="n features", rows=4,
                            style={"description_width": "90px"})
cv_sl = W.IntSlider(value=5, min=3, max=10, description="CV folds",
                    continuous_update=False, style={"description_width": "90px"})
run_btn = W.Button(description="Run grid search", button_style="success", icon="play")
out5 = W.Output()


def run_search(_=None):
    with out5:
        clear_output(wait=True)
        if "pipe" not in S:
            print("Build the pipeline first (Step 4)."); return
        if "X_train" not in S:
            print("Split the data first (Step 3)."); return

        nfeat = sorted(nfeat_dd.value) or [5]
        # the parameter name depends on the selector
        if S["fs_name"] == "SelectFromModel":
            grid_fs = {"FS__max_features": nfeat}
        elif S["fs_name"] == "Univariate (ANOVA F)":
            grid_fs = {"FS__k": nfeat}
        else:
            grid_fs = {"FS__n_features_to_select": nfeat}

        cname = S["clf_name"]
        if cname == "SVM":
            grid_clf = {"clf__C": [0.5, 1.0, 5.0]}
        elif cname == "Neural network":
            grid_clf = {"clf__hidden_layer_sizes": [(20, 20), (10, 10), (5, 5)]}
        else:
            grid_clf = {"clf__max_depth": [5, 10, 20]}

        param_grid = {**grid_fs, **grid_clf}
        n_fits = 1
        for v in param_grid.values():
            n_fits *= len(v)
        print("Grid:", param_grid)
        print("Fitting %d combinations x %d folds = %d model fits...\n"
              % (n_fits, cv_sl.value, n_fits * cv_sl.value))

        t1 = time.perf_counter()
        search = GridSearchCV(S["pipe"], param_grid, n_jobs=-1,
                              cv=int(cv_sl.value), verbose=0)
        fit = search.fit(S["X_train"], S["Y_train"])
        t2 = time.perf_counter()

        S.update(fit=fit, search=search, seconds=t2 - t1, param_grid=param_grid)

        print("Best parameters :", search.best_params_)
        print("Best CV score   : %.3f" % search.best_score_)
        print("Time taken      : %.1f s" % (t2 - t1))


run_btn.on_click(run_search)
display(W.VBox([W.HBox([nfeat_dd, cv_sl]), run_btn, out5]))

## Step 6 — How good is the model?

Two views of the same result:

* the **confusion matrix** counts how many patients ended up in each cell;
* the **ROC curve** sweeps the decision threshold and plots sensitivity against
  1 − specificity. The area under it (AUC) is threshold-independent.

Accuracy alone is misleading on an imbalanced dataset: always read sensitivity
and specificity together.

Everything is reported **twice**: on the training set and on the test set. A model
that scores far better on the data it has already seen is overfitting — that gap
is the single most useful number on this page.

In [ ]:
#@title 📈 Step 6 — Confusion matrix and ROC curve { display-mode: "form" }

eval_btn = W.Button(description="Evaluate on the test set", button_style="primary",
                    icon="bar-chart")
out6 = W.Output()


def evaluate(_=None):
    with out6:
        clear_output(wait=True)
        if "fit" not in S:
            print("Run the grid search first (Step 5)."); return

        best = S["fit"].best_estimator_
        Xte, Yte = S["X_test"], S["Y_test"]
        pred = best.predict(Xte)
        prob = best.predict_proba(Xte)[:, 1]

        acc = accuracy_score(Yte, pred)
        sens = recall_score(Yte, pred)
        spec = recall_score(Yte, pred, pos_label=0)
        fpr, tpr, _ = roc_curve(Yte, prob)
        roc_auc = auc(fpr, tpr)

        # the same metrics on the data the model was trained on
        Xtr, Ytr = S["X_train"], S["Y_train"]
        pred_tr = best.predict(Xtr)
        prob_tr = best.predict_proba(Xtr)[:, 1]
        fpr_tr, tpr_tr, _ = roc_curve(Ytr, prob_tr)
        tr = dict(acc=accuracy_score(Ytr, pred_tr),
                  sens=recall_score(Ytr, pred_tr),
                  spec=recall_score(Ytr, pred_tr, pos_label=0),
                  auc=auc(fpr_tr, tpr_tr))
        te = dict(acc=acc, sens=sens, spec=spec, auc=roc_auc)
        S.update(acc=acc, sens=sens, spec=spec, auc=roc_auc, train_metrics=tr)

        print("                train    test     gap")
        for k, label in [("acc", "Accuracy"), ("sens", "Sensitivity"),
                         ("spec", "Specificity"), ("auc", "AUC")]:
            print("   %-11s %6.3f  %6.3f  %+6.3f" % (label, tr[k], te[k], te[k] - tr[k]))

        gap = tr["auc"] - te["auc"]
        print()
        if gap > 0.15:
            print("The model is clearly overfitting: it is %.2f AUC better on data it has"
                  "\nalready seen. Try fewer features, a shallower model, or more data." % gap)
        elif gap > 0.05:
            print("Mild overfitting (AUC gap %.2f). Common and often acceptable." % gap)
        else:
            print("Train and test agree closely (AUC gap %.2f): the model generalises." % gap)
        print("\nSensitivity + specificity on test = %.2f  (1.0 = useless, 2.0 = perfect)"
              % (sens + spec))

        fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), dpi=110)
        sns.heatmap(confusion_matrix(Yte, pred), annot=True, fmt="d",
                    cmap="Blues", cbar=False, ax=ax[0])
        ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
        ax[0].set_title("Confusion matrix (test set)")

        ax[1].plot(fpr_tr, tpr_tr, color="#1A6FBF", lw=2, ls="--",
                   label="train  AUC = %0.2f" % tr["auc"])
        ax[1].plot(fpr, tpr, color="crimson", lw=2.5,
                   label="test   AUC = %0.2f" % roc_auc)
        ax[1].plot([0, 1], [0, 1], "--", color="grey", lw=1)
        ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
        ax[1].set_xlabel("1 - specificity"); ax[1].set_ylabel("Sensitivity")
        ax[1].set_title("ROC curve"); ax[1].legend(loc="lower right")
        ax[1].grid(linestyle="--", alpha=0.5)

        keys = ["acc", "sens", "spec", "auc"]
        xpos = np.arange(len(keys)); w = 0.38
        ax[2].bar(xpos - w / 2, [tr[k] for k in keys], w, label="train", color="#1A6FBF")
        ax[2].bar(xpos + w / 2, [te[k] for k in keys], w, label="test", color="crimson")
        ax[2].set_xticks(xpos)
        ax[2].set_xticklabels(["acc", "sens", "spec", "AUC"])
        ax[2].set_ylim(0, 1.05); ax[2].legend()
        ax[2].set_title("Train vs test")
        ax[2].grid(axis="y", linestyle="--", alpha=0.5)

        plt.tight_layout(); plt.show()


eval_btn.on_click(evaluate)
display(W.VBox([eval_btn, out6]))

## Step 7 — Which features did the model actually use?

Feature selection kept only a handful of variables. Two things worth checking:

* **importance** — how much each surviving feature contributes to the decision;
* **correlation** — if two selected features are strongly correlated they carry
  the same information, and their individual importances become hard to read.

In [ ]:
#@title 🔬 Step 7 — Selected features: importance and correlation { display-mode: "form" }

interp_btn = W.Button(description="Show selected features", button_style="primary",
                      icon="search")
out7 = W.Output()


def interpret(_=None):
    with out7:
        clear_output(wait=True)
        if "fit" not in S:
            print("Run the grid search first (Step 5)."); return

        best = S["fit"].best_estimator_
        mask = best.named_steps["FS"].get_support()
        names = S["X_train"].columns[mask]
        S["feature_names"] = names
        print("%d features selected out of %d:" % (len(names), S["X_train"].shape[1]))
        print(", ".join(map(str, names)))

        clf_step = best.named_steps["clf"]
        has_imp = hasattr(clf_step, "feature_importances_")

        fig, ax = plt.subplots(1, 2, figsize=(14, 5), dpi=110)
        if has_imp:
            imp = clf_step.feature_importances_
            order = imp.argsort()
            ax[0].barh(np.array(names)[order], imp[order], color="#1A6FBF")
            ax[0].set_xlabel("Feature importance")
            ax[0].grid(axis="x", linestyle="--", alpha=0.5)
        else:
            ax[0].text(0.5, 0.5,
                       "%s does not expose\nfeature importances" % S["clf_name"],
                       ha="center", va="center", fontsize=11)
            ax[0].axis("off")
        ax[0].set_title("Importance of the selected features")

        corr = pd.DataFrame(S["X_train"], columns=S["X_train"].columns)[names].corr()
        sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax[1],
                    annot=len(names) <= 8, fmt=".2f", cbar=True)
        ax[1].set_title("Correlation among selected features")
        plt.tight_layout(); plt.show()

        strong = [(names[i], names[j], corr.iloc[i, j])
                  for i in range(len(names)) for j in range(i + 1, len(names))
                  if abs(corr.iloc[i, j]) > 0.8]
        if strong:
            print("\nStrongly correlated pairs (|r| > 0.8) — redundant information:")
            for a, b, r_ in strong:
                print("   %-22s %-22s r = %+.2f" % (a, b, r_))


interp_btn.on_click(interpret)
display(W.VBox([interp_btn, out7]))

## Step 8 — SHAP: why did the model decide *this* for *this* patient?

Feature importance in Step 7 tells you which variables matter **on average**.
SHAP goes further: it splits each individual prediction into the contribution of
each feature, so you can ask *why was this particular patient classified as
high risk?*

* the **beeswarm** plot shows every patient as a dot: position = how much that
  feature pushed the prediction, colour = whether the feature value was high or low;
* the **waterfall** plot opens up one single patient.

For tree models this is exact and fast. For SVM and neural networks SHAP has to
probe the model thousands of times, so we sample a subset — it takes longer and
is approximate.

In [ ]:
#@title 🧩 Step 8 — SHAP explanations { display-mode: "form" }

try:
    import shap
except ImportError:
    !pip -q install shap
    import shap

patient_sl = W.IntSlider(value=0, min=0, max=0, description="patient",
                         continuous_update=False, style={"description_width": "90px"})
shap_btn = W.Button(description="Explain with SHAP", button_style="primary", icon="puzzle-piece")
out_shap = W.Output()


def run_shap(_=None):
    with out_shap:
        clear_output(wait=True)
        if "fit" not in S:
            print("Run the grid search first (Step 5)."); return

        best = S["fit"].best_estimator_
        model = best.named_steps["clf"]
        mask = best.named_steps["FS"].get_support()
        names = list(S["X_train"].columns[mask])

        # push the data through every step except the classifier
        Xte_t = pd.DataFrame(best[:-1].transform(S["X_test"]), columns=names)
        Xtr_t = pd.DataFrame(best[:-1].transform(S["X_train"]), columns=names)

        is_tree = hasattr(model, "feature_importances_")
        if is_tree:
            explainer = shap.TreeExplainer(model)
            Xshow = Xte_t
            sv = explainer.shap_values(Xshow)
            base = explainer.expected_value
        else:
            print("%s is not a tree model: SHAP has to probe it repeatedly."
                  % S["clf_name"])
            print("Using 30 background samples and the first 25 test patients...\n")
            bg = shap.sample(Xtr_t, min(30, len(Xtr_t)), random_state=0)
            explainer = shap.KernelExplainer(model.predict_proba, bg)
            Xshow = Xte_t.iloc[:25]
            sv = explainer.shap_values(Xshow, nsamples=100, silent=True)
            base = explainer.expected_value

        # normalise to the positive class: shap may return a 3-D array or a list
        arr = np.array(sv)
        if arr.ndim == 3:
            arr = arr[:, :, 1]
        elif isinstance(sv, list):
            arr = np.array(sv[1])
        b = base[1] if isinstance(base, (list, np.ndarray)) and np.ndim(base) > 0 else base

        S.update(shap_values=arr, shap_X=Xshow, shap_base=b, shap_names=names)
        patient_sl.max = len(Xshow) - 1

        shap.summary_plot(arr, Xshow, feature_names=names, show=False,
                          plot_size=(9, 0.45 * len(names) + 1.5))
        plt.title("SHAP: contribution of each feature to the positive class", fontsize=11)
        plt.tight_layout(); plt.show()

        print("\nMove the slider and press the button again to open one patient.")
        show_patient()


def show_patient(_=None):
    if "shap_values" not in S:
        return
    i = min(patient_sl.value, len(S["shap_X"]) - 1)
    expl = shap.Explanation(values=S["shap_values"][i],
                            base_values=S["shap_base"],
                            data=S["shap_X"].iloc[i].values,
                            feature_names=S["shap_names"])
    shap.plots.waterfall(expl, show=False)
    plt.title("Patient #%d — how the prediction was built" % i, fontsize=11)
    plt.tight_layout(); plt.show()


shap_btn.on_click(run_shap)
patient_sl.observe(lambda ch: (out_shap.clear_output(wait=True),
                               run_shap_patient_only()) if ch["name"] == "value" else None,
                   names="value")


def run_shap_patient_only():
    with out_shap:
        clear_output(wait=True)
        if "shap_values" not in S:
            print("Press 'Explain with SHAP' first."); return
        show_patient()


display(W.VBox([W.HBox([shap_btn, patient_sl]), out_shap]))

## Step 9 — Do the selected features separate the two groups?

For each selected feature we compare the two outcome groups with the
**Wilcoxon rank-sum test** (also known as Mann–Whitney U): a non-parametric test
for two *independent* samples of possibly different size.

> Note: the earlier version of this notebook called `scipy.stats.wilcoxon`,
> which is the *signed-rank* test for **paired** data. With two independent
> groups it either crashes or silently pairs unrelated patients. The correct
> function for this comparison is `ranksums`.

In [ ]:
#@title 📉 Step 8 — Group comparison per feature { display-mode: "form" }

from scipy.stats import ranksums

alpha_sl = W.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01,
                         description="alpha", readout_format=".2f",
                         continuous_update=False, style={"description_width": "90px"})
cmp_btn = W.Button(description="Compare groups", button_style="primary", icon="area-chart")
out8 = W.Output()


def compare(_=None):
    with out8:
        clear_output(wait=True)
        if "feature_names" not in S:
            print("Run Step 7 first, so we know which features were selected."); return

        names = list(S["feature_names"])
        df = pd.DataFrame(S["X_train"], columns=S["X_train"].columns)[names]
        y = pd.Series(S["Y_train"]).reset_index(drop=True)
        df = df.reset_index(drop=True)
        N, P = df[y == 0], df[y == 1]

        alpha = alpha_sl.value
        n = len(names)
        fig, axes = plt.subplots(1, n, figsize=(max(8, 2.6 * n), 4.5), dpi=110)
        if n == 1:
            axes = [axes]

        sig = []
        for ax, col in zip(axes, names):
            stat, p = ranksums(N[col], P[col])
            if p < alpha:
                sig.append((col, p))
            bp = ax.boxplot([N[col], P[col]], labels=["neg", "pos"], patch_artist=True,
                            medianprops=dict(color="black"))
            for patch, c in zip(bp["boxes"], ["#9EC5E8", "#F4A582"]):
                patch.set_facecolor(c)
            ax.set_title("%s\np = %.3g%s" % (col, p, " *" if p < alpha else ""),
                         fontsize=9)
            ax.tick_params(labelsize=8)
            ax.grid(axis="y", linestyle="--", alpha=0.4)

        plt.suptitle("Selected features by outcome group (Wilcoxon rank-sum)", y=1.02)
        plt.tight_layout(); plt.show()

        if sig:
            print("Significant at alpha = %.2f:" % alpha)
            for c, p in sorted(sig, key=lambda t: t[1]):
                print("   %-25s p = %.4g" % (c, p))
        else:
            print("No feature reaches significance at alpha = %.2f." % alpha)
        print("\nRemember: these p-values are computed on the training set, after"
              "\nfeature selection picked these very features. Treat them as"
              "\ndescriptive, not as a hypothesis test.")


cmp_btn.on_click(compare)
display(W.VBox([W.HBox([alpha_sl, cmp_btn]), out8]))

## Step 10 — Put several models side by side

So far you changed one model at a time. Here you pick **as many classifiers as
you like** (ctrl-click / cmd-click to multi-select) and the notebook trains them
all with the same scaler, the same feature selection and the same grid search,
on the same split.

That is the honest way to compare: everything is identical except the model.
Expect the differences to be smaller than you think — and often smaller than the
difference between rebalancing on and off.

In [ ]:
#@title 🏁 Step 10 — Train several classifiers and compare { display-mode: "form" }

multi_sel = W.SelectMultiple(options=list(CLASSIFIERS),
                             value=("Random forest", "XGBoost", "SVM"),
                             description="models", rows=6,
                             style={"description_width": "70px"},
                             layout=W.Layout(width="330px"))
cmp_nfeat = W.SelectMultiple(options=[3, 5, 10, 15], value=(5,),
                             description="n feats", rows=4,
                             style={"description_width": "70px"},
                             layout=W.Layout(width="200px"))
cmp_run = W.Button(description="Compare models", button_style="success", icon="flag-checkered")
out10 = W.Output()


def compare_models(_=None):
    with out10:
        clear_output(wait=True)
        if "X_train" not in S:
            print("Split the data first (Step 3)."); return
        chosen = list(multi_sel.value)
        if not chosen:
            print("Select at least one model."); return

        nfeat = sorted(cmp_nfeat.value) or [5]
        Xtr, Ytr, Xte, Yte = S["X_train"], S["Y_train"], S["X_test"], S["Y_test"]
        rows = []
        fig, ax = plt.subplots(figsize=(7, 6), dpi=110)

        for cname in chosen:
            clf = CLASSIFIERS[cname]()
            sel_est = clf
            if cname in ("SVM", "Neural network"):
                sel_est = RandomForestClassifier(n_estimators=100, random_state=0)
            f5 = SelectFromModel(estimator=sel_est)
            steps = []
            if SCALERS[sca_dd.value] is not None:
                steps.append(("scaler", SCALERS[sca_dd.value]()))
            steps += [("FS", f5), ("clf", clf)]
            pipe = Pipeline(steps)

            grid = {"FS__max_features": nfeat}
            if cname == "SVM":
                grid["clf__C"] = [0.5, 1.0]
            elif cname == "Neural network":
                grid["clf__hidden_layer_sizes"] = [(10, 10)]
            else:
                grid["clf__max_depth"] = [5, 10]

            t0 = time.perf_counter()
            try:
                fit = GridSearchCV(pipe, grid, n_jobs=-1, cv=5).fit(Xtr, Ytr)
            except Exception as e:
                print("%-15s failed: %s" % (cname, type(e).__name__)); continue
            dt = time.perf_counter() - t0

            best = fit.best_estimator_
            prob = best.predict_proba(Xte)[:, 1]
            pred = best.predict(Xte)
            fpr, tpr, _ = roc_curve(Yte, prob)
            a = auc(fpr, tpr)
            a_tr = auc(*roc_curve(Ytr, best.predict_proba(Xtr)[:, 1])[:2])
            rows.append(dict(model=cname,
                             AUC_test=round(a, 3), AUC_train=round(a_tr, 3),
                             gap=round(a_tr - a, 3),
                             accuracy=round(accuracy_score(Yte, pred), 3),
                             sensitivity=round(recall_score(Yte, pred), 3),
                             specificity=round(recall_score(Yte, pred, pos_label=0), 3),
                             seconds=round(dt, 1)))
            ax.plot(fpr, tpr, lw=2.2, label="%s (AUC %.2f)" % (cname, a))

        if not rows:
            plt.close(fig); return
        ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("1 - specificity"); ax.set_ylabel("Sensitivity")
        ax.set_title("ROC on the test set — %s, rebalancing: %s"
                     % (S["dataset"], S["balancing"]))
        ax.legend(loc="lower right"); ax.grid(linestyle="--", alpha=0.5)
        plt.tight_layout(); plt.show()

        tab = pd.DataFrame(rows).sort_values("AUC_test", ascending=False)
        display(tab.reset_index(drop=True))
        best_row = tab.iloc[0]
        print("Best on this split: %s (test AUC %.3f)." % (best_row["model"], best_row["AUC_test"]))
        if (tab["AUC_test"].max() - tab["AUC_test"].min()) < 0.05:
            print("All models are within 0.05 AUC of each other — on this dataset the"
                  "\nchoice of model matters less than the data itself.")


cmp_run.on_click(compare_models)
display(W.VBox([W.HBox([multi_sel, cmp_nfeat]), cmp_run, out10]))

## Step 11 — Log your result for the class

Your row goes into the `<dataset>Results` worksheet, next to everybody else's.
Once a few people have submitted, open the sheet and look at what separates the
good runs from the poor ones.

In [ ]:
#@title 📤 Step 9 — Send your result to the shared sheet { display-mode: "form" }

name_txt = W.Text(value="", placeholder="your name or nickname, email",
                  description="name", style={"description_width": "70px"},
                  layout=W.Layout(width="520px"))
notes_txt = W.Text(value="", placeholder="anything you noticed",
                   description="notes", style={"description_width": "70px"},
                   layout=W.Layout(width="520px"))
send_btn = W.Button(description="Send to Google Sheet", button_style="success",
                    icon="upload")
out9 = W.Output()


def send(_=None):
    with out9:
        clear_output(wait=True)
        missing = [k for k in ("fit", "acc") if k not in S]
        if missing:
            print("Run Step 5 and Step 6 first."); return
        if not name_txt.value.strip():
            print("Please write your name first."); return

        row = [
            datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
            "%.3f" % (S["seconds"] / 60.0),
            "%.3f" % S["acc"],
            "%.3f" % S["sens"],
            "%.3f" % S["spec"],
            str(S["pipe"].steps),
            str(S["fit"].best_params_),
            S["balancing"],
            name_txt.value.strip(),
            notes_txt.value.strip(),
        ]

        ws = sht.worksheet(S["dataset"] + "Results")
        ws.append_row(row, value_input_option="USER_ENTERED")

        print("Sent to worksheet '%sResults'." % S["dataset"])
        print("   dataset     :", S["dataset"])
        print("   rebalancing :", S["balancing"])
        print("   pipeline    : %s / %s / %s"
              % (S["scaler_name"], S["fs_name"], S["clf_name"]))
        print("   accuracy %.3f | sensitivity %.3f | specificity %.3f | AUC %.3f"
              % (S["acc"], S["sens"], S["spec"], S["auc"]))


send_btn.on_click(send)
display(W.VBox([name_txt, notes_txt, send_btn, out9]))

## Step 12 — The class leaderboard

Reads the results worksheet back and ranks everybody's submissions. Look at the
top rows and ask what they have in common: the same classifier? rebalancing on
or off? few features or many?

This is the most useful part of the exercise — a small, real meta-analysis of
sixty experiments run by sixty people on the same data.

In [ ]:
#@title 🏆 Step 12 — Class leaderboard { display-mode: "form" }

sort_dd = W.Dropdown(options=["sensitivity + specificity", "accuracy",
                              "sensitivity", "specificity"],
                     value="sensitivity + specificity", description="rank by",
                     style={"description_width": "90px"})
topn_sl = W.IntSlider(value=10, min=5, max=40, description="show top",
                      continuous_update=False, style={"description_width": "90px"})
lb_btn = W.Button(description="Load leaderboard", button_style="primary", icon="trophy")
out12 = W.Output()

COLS = ["timestamp", "minutes", "accuracy", "sensitivity", "specificity",
        "pipeline", "best_params", "rebalancing", "name", "notes"]


def leaderboard(_=None):
    with out12:
        clear_output(wait=True)
        if "dataset" not in S:
            print("Load a dataset first (Step 2)."); return
        ws = sht.worksheet(S["dataset"] + "Results")
        vals = ws.get_all_values()
        if len(vals) < 2:
            print("No results submitted yet for %s." % S["dataset"]); return

        df = pd.DataFrame(vals[1:], columns=COLS[:len(vals[0])])
        for c in ("accuracy", "sensitivity", "specificity"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=["accuracy", "sensitivity", "specificity"])
        if df.empty:
            print("No usable numeric results yet."); return

        df["sens+spec"] = df["sensitivity"] + df["specificity"]
        key = {"sensitivity + specificity": "sens+spec", "accuracy": "accuracy",
               "sensitivity": "sensitivity", "specificity": "specificity"}[sort_dd.value]
        df = df.sort_values(key, ascending=False).reset_index(drop=True)
        df.index += 1

        show = df.head(int(topn_sl.value))[
            ["name", "accuracy", "sensitivity", "specificity", "sens+spec",
             "rebalancing", "notes"]]
        print("%s — %d submissions, ranked by %s\n" % (S["dataset"], len(df), sort_dd.value))
        display(show)

        # what do the good runs have in common?
        top = df.head(max(3, len(df) // 4))
        print("\nAmong the top %d runs:" % len(top))
        for col in ("rebalancing",):
            if col in top and top[col].notna().any():
                vc = top[col].value_counts()
                whole = df[col].value_counts()
                for k, v in vc.items():
                    if not k:
                        continue
                    print("   %-10s %2d of %2d top runs (%d of %d overall)"
                          % (k, v, len(top), whole.get(k, 0), len(df)))

        fig, ax = plt.subplots(figsize=(7, 5), dpi=110)
        colours = {"SMOTE": "#E65100", "ADASYN": "#6B3FA0", "none": "#1A6FBF"}
        for meth, grp in df.groupby("rebalancing"):
            ax.scatter(1 - grp["specificity"], grp["sensitivity"], s=45, alpha=0.75,
                       label=meth or "(blank)", color=colours.get(meth, "grey"))
        ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("1 - specificity"); ax.set_ylabel("Sensitivity")
        ax.set_title("Every submission for %s" % S["dataset"])
        ax.legend(title="rebalancing"); ax.grid(linestyle="--", alpha=0.5)
        plt.tight_layout(); plt.show()


lb_btn.on_click(leaderboard)
display(W.VBox([W.HBox([sort_dd, topn_sl]), lb_btn, out12]))

## What to try next

Change **one** thing at a time and watch the AUC and the sensitivity/specificity
balance:

* switch the **rebalancing** between none / SMOTE / ADASYN on an imbalanced dataset;
* keep the classifier and change only the **number of selected features**;
* compare a tree-based model with **SVM** or a **neural network** (note that the
  feature-importance plot disappears — those models do not expose it);
* run the same configuration on a **different dataset** and see whether your
  conclusions travel.